# 4.10 · KNN 回归 / K-Nearest Neighbors Regression

> **课程定位 / Where this fits**
> 第 10 课，**Part 4 · 监督学习：回归**。
> Lesson 10, **Part 4 · Supervised Regression**.
>
> 前面的模型都"学一个全局函数"。KNN 回归是**非参数惰性**模型：不训练、不假设函数形式，预测时直接找最近的 K 个邻居、取它们目标值的**平均**。它直观、零假设，但有缩放和维度诅咒两大坑。(分类版见 5.3，这里是回归版。)
> Earlier models "learn a global function". KNN regression is **nonparametric and lazy**: no training, no functional-form assumption — at prediction time it finds the K nearest neighbors and averages their targets. Intuitive and assumption-free, but with two pitfalls: scaling and the curse of dimensionality. (Classification version in 5.3; this is regression.)
>
> 💼 **实战/面试视角**："KNN 回归怎么预测 / k 怎么选 / 为什么高维失效" 是基础概念题。
> 💼 **Practical/interview angle:** "how KNN regresses / choosing k / why it fails in high-dim" are basics.

> 💡 **面试相关 / Interview-relevant**
> - "KNN 回归 vs 分类的区别"（出镜率 ★★★，平均 vs 投票）
> - "k 对偏差方差的影响"（★★★★）
> - "为什么 KNN 必须缩放"（★★★★）
> - "维度诅咒为什么让 KNN 失效"（★★★★★）
> - "均匀权重 vs 距离加权"（★★★）

---

## 学习目标 / Learning Objectives

1. 理解 KNN 回归 = 局部邻居目标的平均。
   Understand KNN regression = averaging neighbors' targets locally.
2. **从零**实现并对照 sklearn。
   Implement from scratch and match sklearn.
3. 理解 k 的偏差方差权衡，用 CV 选 k。
   Understand k's bias-variance trade-off; choose k by CV.
4. 实测**维度诅咒**让 KNN 失效。
   Empirically show the curse of dimensionality breaking KNN.
5. 牢记 KNN **必须缩放**。
   Remember KNN must be scaled.

## 目录 / TOC
1. [先建直觉 + 数据](#1)
2. [从零实现 + 阶梯曲线 ⭐](#2)
3. [k 的偏差方差 + 选 k ⭐](#3)
4. [距离加权](#4)
5. [维度诅咒 ⭐](#5)
6. [必须缩放 + 小结 ⭐](#6)


<a id="1"></a>
## 1. 先建直觉 + 数据 / Intuition & Data

KNN 回归的想法和 KNN 分类(5.3)一模一样，只是最后一步从"投票"换成"**平均**"：要预测一个新点，就找训练集里离它最近的 K 个点，**取它们目标值的平均**作为预测。它**不学任何全局函数**——预测完全由"附近的样本长什么样"决定，是纯粹的**局部**方法。
KNN regression is identical to KNN classification (5.3), except the last step swaps "vote" for "**average**": to predict a new point, find its K nearest training points and **average their targets**. It **learns no global function** — the prediction is entirely determined by "what nearby samples look like", a purely **local** method.

我们用 **Diamonds**（钻石数据，seaborn 内置）：用克拉、尺寸等数值特征预测价格。
We use **Diamonds** (built into seaborn): predict price from carat, dimensions, etc.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsRegressor
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

df = sns.load_dataset("diamonds").sample(5000, random_state=0).reset_index(drop=True)
X = df[["carat","depth","table","x","y","z"]].values
y = df["price"].values
print(f"Diamonds: {X.shape}, 预测价格 price")
print(df[["carat","price"]].describe().round(2).T)


<a id="2"></a>
## 2. 从零实现 + 阶梯曲线 ⭐ / From Scratch & the Staircase

代码就是定义的直译：算距离 → 取最近 k 个 → 求它们 y 的平均。用单特征（carat→price）画出来，能看到 KNN 的标志性外观：**阶梯状/锯齿状曲线**——因为它是分段局部平均，没有平滑的函数形式。
The code is a direct translation: compute distances → take the k nearest → average their y. Plotting on one feature (carat→price) reveals KNN's signature look: a **staircase/jagged curve** — it's a piecewise local average with no smooth functional form.


In [ ]:
def knn_predict(X_train, y_train, X_query, k=5):
    preds = []
    for xq in X_query:
        dists = np.sqrt(((X_train - xq)**2).sum(axis=1))   # 到所有训练点的欧氏距离
        nn = np.argsort(dists)[:k]                          # 取距离最小的 k 个的索引
        preds.append(y_train[nn].mean())                    # 这 k 个邻居的目标均值
    return np.array(preds)

# 单特征 carat→price 便于可视化 / 1-feature demo
xc = df["carat"].values.reshape(-1, 1)
order = np.argsort(xc.ravel()); xc, yc = xc[order], y[order]    # 按 carat 排序便于画线
x_plot = np.linspace(xc.min(), xc.max(), 200).reshape(-1, 1)

sk_pred = KNeighborsRegressor(n_neighbors=5).fit(xc, yc).predict(x_plot)
my_pred = knn_predict(xc, yc, x_plot, k=5)
print(f"从零 vs sklearn 最大差异: {np.abs(my_pred - sk_pred).max():.4f} → 一致")

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(xc, yc, alpha=0.15, s=8, label="数据 data")
ax.plot(x_plot, my_pred, "r-", lw=2, label="KNN (k=5)")
ax.set_xlabel("carat"); ax.set_ylabel("price"); ax.legend()
ax.set_title("KNN 回归: 阶梯状曲线(每段=局部邻居平均)")
plt.tight_layout(); plt.show()
print("KNN 曲线呈阶梯/锯齿状 — 因为是局部平均, 没有平滑的函数形式")


<a id="3"></a>
## 3. k 的偏差方差 + 选 k ⭐ / k's Bias-Variance & Choosing k

k 是 KNN 的核心旋钮，控制复杂度（同 5.3）：
k is KNN's core knob, controlling complexity (as in 5.3):
- **k=1**：预测完全跟着最近那一个点 → 曲线锯齿、**过拟合（高方差）**。
  **k=1:** prediction follows the single nearest point → jagged, **overfit (high variance)**.
- **k 很大**：平均一大片邻居 → 曲线趋平、**欠拟合（高偏差）**，k=n 时退化成"永远预测全局均值"。
  **large k:** averages many neighbors → flattens, **underfit (high bias)**; at k=n it predicts the global mean.

用交叉验证选中等的 k。
Choose a moderate k by cross-validation.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, k in zip(axes, [1, 20, 500]):
    pred = KNeighborsRegressor(n_neighbors=k).fit(xc, yc).predict(x_plot)
    ax.scatter(xc, yc, alpha=0.1, s=6); ax.plot(x_plot, pred, "r-", lw=2)
    tag = "过拟合(追噪声)" if k==1 else ("刚好" if k==20 else "欠拟合(过平滑)")
    ax.set_title(f"k={k}  {tag}")
plt.tight_layout(); plt.show()

# CV 选 k / pick k by CV
Xc_full = df[["carat"]].values
ks = [1, 3, 5, 10, 20, 50, 100, 200]
cv = [cross_val_score(KNeighborsRegressor(n_neighbors=k), Xc_full, y, cv=5, scoring="r2").mean() for k in ks]
print(f"各 k 的 CV R²: {dict(zip(ks, np.round(cv,3)))}")
print(f"最优 k = {ks[int(np.argmax(cv))]}  (k=1 过拟合, k 太大欠拟合, CV 选中等 k)")


<a id="4"></a>
## 4. 距离加权 / Distance Weighting

默认所有邻居等权平均（`weights='uniform'`）。改成 `weights='distance'`，让**越近的邻居权重越大**（权 ∝ 1/距离）——通常略好，且预测曲线更平滑（近邻主导，过渡更连续）。
By default all neighbors are weighted equally (`weights='uniform'`). Switching to `weights='distance'` makes **closer neighbors count more** (weight ∝ 1/distance) — usually slightly better, with a smoother curve (near neighbors dominate, transitions are more continuous).


In [ ]:
Xc_full = df[["carat"]].values
uniform  = cross_val_score(KNeighborsRegressor(20, weights="uniform"),  Xc_full, y, cv=5, scoring="r2").mean()
distance = cross_val_score(KNeighborsRegressor(20, weights="distance"), Xc_full, y, cv=5, scoring="r2").mean()
print(f"均匀权重 uniform:  CV R² = {uniform:.3f}")
print(f"距离加权 distance: CV R² = {distance:.3f}  (近邻影响更大, 通常略好+更平滑)")

fig, ax = plt.subplots(figsize=(7, 3.5))
for w, c in [("uniform","orange"), ("distance","red")]:
    p = KNeighborsRegressor(20, weights=w).fit(xc, yc).predict(x_plot)
    ax.plot(x_plot, p, color=c, lw=2, label=f"weights={w}")
ax.scatter(xc, yc, alpha=0.08, s=6); ax.legend(); ax.set_title("距离加权 vs 均匀 (k=20)")
plt.tight_layout(); plt.show()


<a id="5"></a>
## 5. 维度诅咒 ⭐ / Curse of Dimensionality

KNN 在高维会**彻底失效**，原因是：维度一高，**所有点之间的距离都趋于相等**——"最近邻"和"最远邻"差不多远，"最近"就失去了意义。下面实测：随维度增加，"最近邻距离 / 最远邻距离"趋向 1。
KNN **breaks down completely** in high dimensions because: as dimensionality grows, **all pairwise distances become nearly equal** — the nearest and farthest neighbors are about as far, so "nearest" loses meaning. Below: as dimension grows, "nearest/farthest distance" approaches 1.


In [ ]:
dims = [1, 2, 5, 10, 50, 100, 500]
ratios = []
for d in dims:
    pts = rng.uniform(0, 1, (1000, d)); q = rng.uniform(0, 1, (1, d))
    dists = np.sqrt(((pts - q)**2).sum(axis=1))
    ratios.append(dists.min() / dists.max())     # 越接近 1 越糟(近邻≈远邻)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(dims, ratios, "o-", lw=2); ax.set_xscale("log")
ax.set_xlabel("维度 dimension d"); ax.set_ylabel("最近邻距离 / 最远邻距离")
ax.set_title("维度诅咒: 高维下近邻和远邻距离趋同 → KNN 失效")
plt.tight_layout(); plt.show()
print("d=1: 近邻比远邻近很多(比值小, KNN 有效)")
print("d=500: 近邻≈远邻(比值≈1, '最近邻'无意义) → KNN 在高维彻底失效")
print("→ 高维必须先降维(PCA, 0.7/Part6) 或改用树/线性模型")


<a id="6"></a>
## 6. 必须缩放 + 小结 ⭐ / Scaling & Summary

KNN 靠距离，所以**必须缩放**：Diamonds 里 carat 在 [0,5]，而 x/y/z 在 [0,10]，量纲不一致会让大量纲特征主导距离。缩放后表现明显改善。
KNN lives on distance, so **scaling is mandatory**: in Diamonds carat is in [0,5] while x/y/z are in [0,10], so mismatched scales let large-scale features dominate. Scaling clearly improves performance.


In [ ]:
from sklearn.pipeline import make_pipeline
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)
raw    = KNeighborsRegressor(10).fit(X_tr, y_tr).score(X_te, y_te)                          # 不缩放
scaled = make_pipeline(StandardScaler(), KNeighborsRegressor(10)).fit(X_tr, y_tr).score(X_te, y_te)  # 缩放(Pipeline 防泄漏)
print(f"KNN 不缩放 unscaled: test R² = {raw:.3f}")
print(f"KNN 缩放后 scaled:   test R² = {scaled:.3f}")
print("缩放是 KNN 的必需品(3.4): 量纲不一时大特征主导距离 → 必须先标准化")


```
KNN 回归: 找最近 k 个邻居, 取它们 y 的平均(分类是投票 5.3); 非参数惰性, 无函数形式
预测曲线呈阶梯/锯齿(局部平均); k=1 过拟合(高方差), k 大欠拟合(高偏差), CV 选 k
距离加权(weights='distance'): 近邻权重大 → 略好+更平滑
维度诅咒: 高维下近邻≈远邻, "最近"失效 → 先降维或换模型
必须缩放(距离对量纲敏感); 训练 O(1), 预测 O(nd) 慢(大数据不友好)
```

### 💡 面试速查 / Interview cheat-sheet
1. **回归取邻居均值, 分类取邻居投票**; 都非参数惰性。
   Regression averages neighbors, classification votes; both nonparametric and lazy.
2. **k 小过拟合(锯齿), k 大欠拟合(平线)**; CV 选 k。
   Small k overfits (jagged), large k underfits (flat); choose by CV.
3. **必须缩放**(距离对量纲敏感)。
   Must scale (distance is scale-sensitive).
4. **维度诅咒**: 高维下近邻≈远邻 → KNN 失效, 先降维。
   Curse of dimensionality: high-dim makes neighbors equidistant → reduce dims first.
5. **预测慢 O(nd)**, 大数据不友好(对比树/线性)。
   Slow prediction O(nd); not for big data (vs trees/linear).

### 下一节 / Next
**4.11 决策树回归**——从惰性局部方法转到树模型: 把空间切成方块, 每块预测一个常数; 可解释、不需缩放, 是随机森林/GBDT 的基石。
**4.11 Decision Tree Regression** — from lazy local methods to trees: split space into boxes, predict a constant per box; interpretable, scaling-free, the building block of forests/GBDT.
